# GPT & Decoder Models: Writing One Word at a Time

**Module 4 · Lesson 4.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- CLM - How GPT Learns (Causal Language Modeling)
- The Autoregressive Loop - Build It From Scratch
- Temperature - The Creativity Dial
- Top-k and Top-p - Smart Filtering Before Picking
- Putting It All Together - Generate Text With Real GPT
- Same Concepts, Modern API - Gemini

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic numpy openai torch -q google-genai transformers

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

## The Single Idea Behind All Text Generation

Every text-generating AI in the world - ChatGPT, Gemini, Claude, LLaMA, Copilot - works on one simple loop:

> ✍️ **Analogy**
>
> **Imagine you're writing a story, but with a strange rule:** you can only write *one word at a time*, and after each word you must stop, re-read everything you've written so far, and then decide the single best next word.
> *"The"* → hmm, what comes next? → *"cat"* → re-read "The cat", what's next? → *"sat"* → re-read "The cat sat", what's next? → *"on"* → ...
> That's it. That's how GPT-5, Claude, and Gemini write an entire essay, poem, or piece of code. **One. Word. At. A. Time.**

Let's watch this happening visually. Here's GPT generating the sentence "The cat sat on the mat":

Round 1: GPT sees "The" and predicts next word

Round 2: GPT sees "The cat" and predicts next word

Round 3: GPT sees "The cat sat" and predicts next word

...and so on until it decides to stop.

This process has a fancy name: **autoregressive generation**. "Auto" = self, "regressive" = feeds back into itself. Each generated word becomes input for the next prediction.

> ℹ️ **Info**
>
> What you'll learn in this lesson
> - **CLM** - how GPT learns (predict the next word, but can only look backwards)
> - **Autoregressive loop** - the generate-one-word-at-a-time loop, coded from scratch
> - **Temperature** - the "creativity dial", built from scratch with pure Python
> - **Top-k and Top-p** - two clever tricks for better text generation
> - **Hands-on** - generate text with a real GPT model using Hugging Face

### By the end of this lesson, you can

- **Explain** how a decoder generates text - causal language modelling plus the autoregressive loop that feeds each new token back in.

- **Build** the decoding controls from scratch (temperature, top-k, top-p) and predict how each one reshapes the next-token distribution.

- **Evaluate** which model and decoding settings fit a task across the 2026 decoder landscape - open vs closed, standard vs reasoning.

- **Compare** the GPT (decoder) and BERT (encoder) families and say when each one wins.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

---

## 🎯 Concept 1: CLM - How GPT Learns (Causal Language Modeling)

### CLM - How GPT Learns (Causal Language Modeling)

BERT guesses hidden words. GPT guesses the NEXT word. That's the key difference.

> 📜 **Analogy**
>
> **Think of a WhatsApp group chat.** You read messages from the top, and you can *predict* what someone will say next based on what's already been said. But you can't see future messages - you only see what's above.
> BERT = reads the ENTIRE chat (past AND future). Great for *understanding*.
> GPT = reads only messages so far (past only). Great for *writing the next message*.

During training, GPT sees a sentence and tries to predict each word using **only the words before it**:

This is called **Causal Language Modeling (CLM)**. "Causal" means "can only look at what came before" - the future is hidden. Let's see it in code:

**📝 `part1_clm_concept.py`**

In [ ]:
# =============================================
# CLM: Causal Language Modeling
# Given previous words → predict the next word.
# This is how GPT-5, Gemini, Claude all learn.
# =============================================

import numpy as np

# Imagine a tiny vocabulary
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran"]

# A very simple "language model" that just counts patterns
# After seeing "the", what usually comes next?
next_word_probs = {
    "the": {"cat": 0.4, "dog": 0.3, "mat": 0.3},
    "cat": {"sat": 0.6, "ran": 0.4},
    "dog": {"sat": 0.3, "ran": 0.7},
    "sat": {"on": 0.9, "the": 0.1},
    "on":  {"the": 0.95, "mat": 0.05},
}

# Generate text one word at a time!
current = "the"
sentence = [current]

print("Generating text word-by-word:\n")
for _ in range(5):
    if current not in next_word_probs:
        break
    
    options = next_word_probs[current]
    words = list(options.keys())
    probs = list(options.values())
    
    # Show the probabilities
    print(f"  After '{current}', options:")
    for w, p in zip(words, probs):
        bar = "#" * int(p * 20)
        print(f"    {w:6s} {p:.0%} {bar}")
    
    # Pick the next word randomly based on probabilities
    next_word = np.random.choice(words, p=probs)
    sentence.append(next_word)
    current = next_word
    print(f"  → Picked: '{next_word}'\n")

print(f"Generated: {' '.join(sentence)}")
print("\nRun it again - you'll get a different sentence!")
print("That's CLM: predict next word, pick one, repeat.")

# Output:
#   After 'the': cat 40% | dog 30% | mat 30%  -> picked 'cat'
#   Generated: the cat sat on the mat

#### 💡 What just happened

What just happened?
  We built a tiny language model using a dictionary of probabilities. After "the", it knows "cat" is roughly 40% likely and "dog" is 30% likely. It picks one randomly (weighted by probability), then uses THAT word to pick the next. This is the *exact same loop* that GPT-5 (and every frontier model) uses - just with a neural network instead of a dictionary, and 100,000+ words instead of 7.

---

## 🎯 Concept 2: The Autoregressive Loop - Build It From Scratch

### The Autoregressive Loop - Build It From Scratch

The exact word-by-word generation process that powers every chatbot.

> 🎲 **Analogy**
>
> **Think of a chain of dominoes.** You push the first one (the prompt). It knocks over the second (first generated word). The second knocks over the third. Each domino only falls because of the one before it. That's autoregressive generation - each word is caused by all the words before it.

Let's build this with a real model. We'll use GPT-2 (free, runs on any laptop) from Hugging Face:

**📝 `part2_autoregressive.py - The generation`** - *loop*

In [ ]:
# =============================================
# THE AUTOREGRESSIVE LOOP
# This is EXACTLY how ChatGPT generates text.
# We'll do it step by step so you can see
# each word being chosen.
# =============================================

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 (small version, runs on any laptop)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# Our starting text (the "prompt")
prompt = "Once upon a time, there was a"

# Convert text to token IDs
input_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'\n")
print("Generating one word at a time:\n")

# Generate 20 new words, one at a time
for step in range(20):
    with torch.no_grad():
        # GPT looks at ALL previous tokens
        outputs = model(input_ids)
        
        # Get probabilities for the NEXT token only
        next_token_logits = outputs.logits[0, -1, :]  # last position
        probs = torch.softmax(next_token_logits, dim=-1)
        
        # Pick the most likely token (greedy)
        next_token_id = probs.argmax().unsqueeze(0).unsqueeze(0)
        next_word = tokenizer.decode(next_token_id[0])
        
        # Show what's happening
        confidence = probs.max().item()
        print(f"  Step {step+1:2d}: '{next_word}' ({confidence:.0%} confident)")
        
        # Add this token to the input for the next round
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)

# Show the full generated text
full_text = tokenizer.decode(input_ids[0])
print(f"\nFull output:\n  '{full_text}'")

# Output:
#   Step  1: ' little' (9% confident) ... Step 20: ' day' (14% confident)
#   Full output: 'Once upon a time, there was a little girl who lived ...'

#### 💡 What just happened

What just happened?
  We manually ran the autoregressive loop that GPT uses: (1) Feed all previous words into the model. (2) Model outputs probabilities for the next word. (3) Pick the most likely word. (4) Add it to the input. (5) Repeat from step 1. Each step, the model sees one more word than before. After 20 rounds, we have 20 new words. That's text generation!

> 💡 **Info**
>
> Why "greedy" picking is boring
> In the code above, we always picked the *most likely* word. This is called **greedy decoding**. The problem? It produces repetitive, predictable text. Ask it to write a story and you'll get "and then he went to the store and then he went to the store and then..." Real chatbots need *some randomness* to be creative. That's where temperature comes in!

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Temperature - The Creativity Dial

### Temperature - The Creativity Dial

Turn it down for facts. Turn it up for creativity.

> 🌶️ **Analogy**
>
> **Think of ordering food at a restaurant.**
> **Low temperature (0.1) =** You always order your usual butter chicken. Safe, predictable, exactly what you expect.
> **Medium temperature (0.7) =** You usually order butter chicken, but sometimes you try the biryani or paneer tikka. A nice mix of reliable and adventurous.
> **High temperature (1.5) =** You randomly pick anything on the menu, including things you've never tried. Exciting but risky - you might get something amazing or something terrible.
> **Where the analogy breaks down:** a diner picks from a fixed menu, but the model re-scores all ~100,000 tokens from scratch at every single step.

Mathematically, temperature is *shockingly simple*. You just **divide the scores by the temperature number** before converting to probabilities. That's it. Let's see:

**📝 `part3_temperature.py - Built from`** - *scratch*

In [ ]:
# =============================================
# TEMPERATURE FROM SCRATCH
# It's just ONE line of math. Let's see the
# dramatic effect it has on word choice.
# =============================================

import numpy as np

def softmax(logits):
    """Convert scores to probabilities."""
    e = np.exp(logits - np.max(logits))
    return e / e.sum()

def apply_temperature(logits, temperature):
    """THE ENTIRE TRICK: divide by temperature, then softmax."""
    return softmax(logits / temperature)  # ← That's it. One division.

# GPT's raw scores for the next word after "The cat sat on the"
words  = ["mat", "floor", "roof", "moon", "pizza"]
logits = np.array([5.0,   4.2,    2.5,   0.8,   -1.0])

# Try different temperatures
print("Next word after 'The cat sat on the ___'\n")

for temp in [0.1, 0.5, 1.0, 2.0]:
    probs = apply_temperature(logits, temp)
    
    # What kind of temperature is this?
    if temp <= 0.3:   mood = "Very safe (always picks 'mat')"
    elif temp <= 0.7: mood = "Balanced (usually 'mat', sometimes 'floor')"
    elif temp <= 1.2: mood = "Creative (could pick 'roof' or 'moon')"
    else:             mood = "Wild! (might even pick 'pizza')"
    
    print(f"  Temperature = {temp}")
    print(f"  {mood}")
    for w, p in zip(words, probs):
        bar = "#" * int(p * 40)
        print(f"    {w:8s} {p:5.1%} {bar}")
    print()

# What you'll see:
# temp=0.1 → "mat" gets ~100%, everything else ~0%
# temp=0.5 → "mat" ~83%, "floor" ~17%, others tiny
# temp=1.0 → "mat" ~65%, "floor" ~29%, "roof" ~5%
# temp=2.0 → flatter: "mat" ~47%, "floor" ~32%, "roof" ~14%, even "pizza" ~2%

# Output:
#   temp=0.1 -> mat 100.0%   |   temp=1.0 -> mat 64.6%, floor 29.0%, roof 5.3%
#   temp=2.0 -> flatter: mat 47.0%, floor 31.5%, roof 13.5%, pizza 2.3%

#### 💡 What just happened

What just happened?

  Temperature = **dividing the scores by a number before softmax**. That's the entire trick.  
  

  • **Small number (0.1)** = scores get multiplied by 10, differences become HUGE, top word wins almost 100%  

  • **Large number (2.0)** = scores get divided by 2, differences shrink, the gaps close sharply and rare words get a real shot  
  

  One line of math. Dramatic effect on creativity.

> ✅ **Info**
>
> When to use which temperature
> - **0.0 - 0.3** - Factual answers, JSON output, code generation. You want the "right" answer every time.
> - **0.5 - 0.7** - Conversational responses, customer support. Reliable but not robotic. *Most common setting.*
> - **0.8 - 1.2** - Creative writing, brainstorming, poetry. You want variety and surprise.
> - **1.5+** - Experimental. Often produces nonsensical text. Rarely useful in production.

---

## 🎯 Concept 4: Top-k and Top-p - Smart Filtering Before Picking

### Top-k and Top-p - Smart Filtering Before Picking

Don't just control randomness. Also filter out the garbage options first.

> 🍱 **Analogy**
>
> **You're at a buffet with 100 dishes.**
> **Top-k = "Only consider the best 5 dishes."** Ignore the other 95. Pick randomly from your top 5.
> **Top-p = "Keep adding dishes until you've covered 90% of your appetite."** If dish 1 covers roughly 40%, dish 2 covers 30%, dish 3 covers 15% - that's about 85%. Add dish 4 (8%) to reach 93%. Stop. Pick from those 4.
> Both prevent the model from picking crazy options like "pizza" after "The cat sat on the".

**📝 `part4_sampling.py - Top-k and Top-p from`** - *scratch*

In [ ]:
# =============================================
# TOP-K and TOP-P SAMPLING
# Filter out the bad options, THEN pick randomly
# from the remaining good ones.
# =============================================

import numpy as np

words = ["mat", "floor", "roof", "moon", "pizza", "quantum", "banana"]
probs = np.array([0.35, 0.28, 0.18, 0.10, 0.05, 0.03, 0.01])

print("Original probabilities:")
for w, p in zip(words, probs):
    print(f"  {w:10s} {p:.0%} {'#' * int(p*30)}")

# ── TOP-K: Keep only the best K options ──
def top_k(words, probs, k=3):
    """Keep only the top k highest-probability words."""
    top_indices = np.argsort(probs)[-k:]  # indices of top k
    mask = np.zeros_like(probs)
    mask[top_indices] = probs[top_indices]
    mask /= mask.sum()  # re-normalize
    return mask

print("\nAfter Top-k = 3 (keep best 3, throw away rest):")
tk = top_k(words, probs, k=3)
for w, p in zip(words, tk):
    marker = " <-- kept" if p > 0 else " (removed)"
    print(f"  {w:10s} {p:.0%}{marker}")

# ── TOP-P: Keep words until cumulative prob reaches P ──
def top_p(words, probs, p=0.85):
    """Keep adding words until their total probability reaches p."""
    sorted_idx = np.argsort(probs)[::-1]  # best first
    cumulative = 0
    keep = []
    for idx in sorted_idx:
        keep.append(idx)
        cumulative += probs[idx]
        if cumulative >= p:
            break
    mask = np.zeros_like(probs)
    mask[keep] = probs[keep]
    mask /= mask.sum()
    return mask

print(f"\nAfter Top-p = 0.85 (keep until 85% covered):")
tp = top_p(words, probs, p=0.85)
cumul = 0
for w, p in zip(words, tp):
    if p > 0:
        cumul += p
        print(f"  {w:10s} {p:.0%}  (running total: {cumul:.0%})")
    else:
        print(f"  {w:10s}  -- (removed)")

# ── PICK A WORD using the filtered probabilities ──
print("\nSampling 10 words with Top-p = 0.85:")
for _ in range(10):
    choice = np.random.choice(words, p=tp)
    print(f"  → {choice}")

# Output:
#   Top-k=3 keeps mat/floor/roof; Top-p=0.85 keeps mat/floor/roof/moon
#   Sampling 10x with Top-p: mat, floor, mat, roof, mat, floor, ...

#### 💡 What just happened

What just happened?
**Top-k:** We kept only the 3 best words and removed the rest. "pizza", "quantum", "banana" are gone - impossible to pick them now.  
  

**Top-p:** We kept adding words (best first) until their combined probability reached roughly 85%. "mat" (35%) + "floor" (28%) + "roof" (18%) = about 81%. Add "moon" (10%) = 91%. Stop. Only these 4 survive.  
  

  Both methods prevent the model from picking absurd words while still allowing some creativity among the good options.

> ✅ **Info**
>
> The modern knob: min-p (2024)
> **Top-p has a weakness:** at high temperature the nucleus balloons and lets junk back in. **Min-p** fixes this with a floor set *relative to the top token*: keep any token whose probability is at least `min_p x (probability of the best token)`. When the model is confident the floor is high and few tokens survive; when it is unsure, more do. It stays stable across temperatures where top-p falls apart.
> **2026 practical guidance:** Temperature + **min-p** for open-source / local models; Temperature + **top-p** for commercial APIs (where min-p may not be exposed); and leave **reasoning models at their locked defaults**. Source: Min-p Sampling, [arXiv 2407.01082](https://arxiv.org/abs/2407.01082).

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 💡 **Info**
>
> Common mistake: the wrong knob for the job
> **What NOT to do:** crank temperature to 2.0 (or a wide top-p) for a task that needs one exact answer. Ask "how many r's are in strawberry?" at T=2.0 and the model will cheerfully sample "2", "4", or a little essay - the creativity dial is actively hurting you. For counting, extraction, JSON, or code, pull temperature DOWN to 0.0-0.3 instead, and save the high settings for brainstorming and fiction.
> A second anti-pattern: hand-tuning temperature on a **reasoning model**. It ships with locked sampling defaults and ignores your value, so a "creative" temperature does nothing but mislead the next person reading your code.

---

## 🎯 Concept 5: Putting It All Together - Generate Text With Real GPT

### Putting It All Together - Generate Text With Real GPT

Use Hugging Face to generate text with different settings and see the differences.

**📝 `part5_generate_stories.py - See the`** - *magic!*

In [ ]:
# =============================================
# GENERATE TEXT with different settings
# Same prompt, different parameters → 
# completely different outputs!
# =============================================

from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

settings = [
    {"name": "Greedy (boring but safe)",
     "args": {"do_sample": False}},
    
    {"name": "Low temperature (factual)",
     "args": {"do_sample": True, "temperature": 0.3, "top_k": 50}},
    
    {"name": "Medium temperature (balanced)",
     "args": {"do_sample": True, "temperature": 0.7, "top_p": 0.9}},
    
    {"name": "High temperature (creative)",
     "args": {"do_sample": True, "temperature": 1.2, "top_p": 0.95}},
    
    {"name": "Very high temperature (wild!)",
     "args": {"do_sample": True, "temperature": 1.8, "top_k": 100}},
]

print(f"Prompt: '{prompt}'\n")
print("=" * 60)

for s in settings:
    output = model.generate(
        input_ids,
        max_new_tokens=40,
        pad_token_id=tokenizer.eos_token_id,
        **s["args"]
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print(f"\n{s['name']}:")
    print(f"  {text}\n")
    print("-" * 60)

print("""
Notice how:
  - Greedy: repetitive, safe, always the same output
  - Low temp: coherent and focused, slight variation
  - Medium: good balance of creativity and coherence
  - High: more creative but sometimes odd word choices
  - Very high: often produces nonsense or bizarre tangents

This is why prompt engineering matters - even the SAME model
gives wildly different results with different settings!
""")

# Output:
#   Greedy:  'The future of artificial intelligence is uncertain. The future ...'
#   Balanced:'The future of artificial intelligence is collaborative, helping ...'

#### 💡 What just happened

What just happened?
  We gave GPT-2 the exact same prompt 5 times with different generation settings. Same model, same prompt, completely different outputs. Temperature controls how "spicy" the word choices are. Top-k and top-p control how many options the model considers. This is why understanding these parameters matters - they're the knobs you'll tune in every GenAI application you build. The tradeoff: GPT-2 is tiny and free but weak - learn the knobs on it, then apply them to a stronger model instead.

---

## 🎯 Concept 6: Same Concepts, Modern API - Gemini

### Same Concepts, Modern API - Gemini

Now that you understand what's under the hood, use a production API.

GPT-2 is great for learning, but modern apps use API-based models. Here's how temperature and top-p work with the Gemini API you'll use throughout this course:

**📝 `part6_gemini_generation.py`**

In [ ]:
# =============================================
# SAME CONCEPTS WITH GEMINI API
# Now you know what these parameters DO under
# the hood - use them with confidence!
# =============================================

import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

prompt = "Write a one-paragraph story about a robot learning to cook."

configs = [
    ("Safe & factual",     0.2, 0.5),   # (name, temperature, top_p)
    ("Balanced",           0.7, 0.9),
    ("Creative",           1.2, 0.95),
]

for name, temp, top_p in configs:
    response = client.models.generate_content(
        model="gemini-3-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temp,       # You now know: logits / temp
            top_p=top_p,            # You now know: keep until cumulative prob
            max_output_tokens=150,
        ),
    )

    print(f"\n{'='*50}")
    print(f"{name} (temp={temp}, top_p={top_p})")
    print(f"{'='*50}")
    print(response.text)

print("""
You now understand EXACTLY what these parameters do:

  temperature=0.2 → divides scores by 0.2 → top word dominates
  temperature=1.2 → divides scores by 1.2 → more words get a chance
  top_p=0.5 → only considers words covering 50% of probability
  top_p=0.95 → considers words covering 95% of probability

You're no longer blindly copy-pasting parameters.
You KNOW what they do. That's the power of understanding.
""")

# Output:
#   Safe & factual -> a measured, literal paragraph about the robot
#   Creative       -> a vivid, surprising paragraph with twists

#### 💡 What just happened

What just happened?

  The same temperature and top-p concepts you built from scratch in Steps 3-4 work identically in production APIs. The only difference: GPT-2 runs locally (the `gpt2` checkpoint is 124M params; `gpt2-xl` is the 1.5B one), while Gemini runs in the cloud (paid, far larger). **Every GenAI API you'll ever use - OpenAI, Anthropic, Gemini, Mistral - has these exact same knobs. You now understand all of them. When to use which: reach for a local model like GPT-2 for offline experiments, and a hosted API instead when you need quality and scale.**

---

## 🎯 Concept 7: The GPT Family - From 117M Parameters to Multimodal

### The GPT Family - From 117M Parameters to Multimodal

Each generation added a key insight. Understanding the evolution explains why ChatGPT works.

**📝 `gpt_evolution.py`** - *History*

In [ ]:
# =============================================
# THE GPT FAMILY - What changed at each step
# =============================================

gpt_family = [
    ("GPT-1",    2018, "117M",   "Proved: decoder pre-training + fine-tuning works"),
    ("GPT-2",    2019, "1.5B",   "Proved: scale → zero-shot abilities (no fine-tuning!)"),
    ("GPT-3",    2020, "175B",   "Proved: in-context learning (few-shot from prompt)"),
    ("InstructGPT",2022,"~1.3B", "Added RLHF → follows instructions, less harmful"),
    ("ChatGPT",  2022, "~20B?",  "InstructGPT + dialogue fine-tuning → the product"),
    ("GPT-4",    2023, "~1.8T?", "MoE architecture, multimodal (text + images)"),
    ("GPT-4o",   2024, "smaller","Omni: text + images + audio, 2x cheaper, 2x faster"),
    ("o1",       2024, "unknown","Reasoning: a hidden chain-of-thought before answering"),
    ("o3/o4-mini",2025,"unknown","Test-time compute scaling; tools inside the reasoning"),
    ("GPT-5",    2025, "unknown","Unified: routes between fast and thinking modes"),
]

print(f"{'Model':<14} {'Year':>5} {'Params':>8} Key Innovation")
print("-" * 75)
for name, year, params, innovation in gpt_family:
    print(f"  {name:<12} {year:>5} {params:>8} {innovation}")

print("""
THE KEY TRANSITIONS:
  GPT-1 → GPT-2:  Scale unlocks zero-shot abilities (no fine-tuning needed)
  GPT-2 → GPT-3:  In-context learning (put examples IN the prompt)
  GPT-3 → ChatGPT: RLHF alignment (Module 9) makes it follow instructions
  ChatGPT → GPT-4: MoE + multimodal (images, audio) → Module 6
  GPT-4 → o1/o3:   Reasoning - spend compute at inference (think before answering)
  o3 → GPT-5:      Unify - one model routes between fast replies and deep thinking

The training pipeline (preview of Module 9):
  1. Pre-training:   Predict next token on internet text ($$$$)
  2. SFT:            Fine-tune on instruction-response pairs
  3. RLHF/DPO:       Human preference alignment → helpful, harmless, honest
  4. Reasoning:       RL on verified traces (o1, o3, GPT-5, DeepSeek-R1)
""")

# Output:
#   GPT-1  2018   117M  Proved decoder pre-training works
#   GPT-5  2025   unknown  Unified: routes fast vs thinking

#### 💡 What just happened

What just happened?
**Each GPT generation added one key insight:** GPT-1 proved pre-training works. GPT-2 proved scale unlocks zero-shot. GPT-3 proved in-context learning. InstructGPT/ChatGPT proved RLHF alignment. GPT-4 proved MoE + multimodal. o1/o3 proved test-time compute. The training pipeline - pre-train → SFT → RLHF → reasoning - is covered in Module 9. Understanding this history explains WHY modern APIs work the way they do. The tradeoff each era faced: more scale and alignment cost more compute and data, which is why a smaller well-aligned or reasoning model is often the better buy today.

> 📦 **Info**
>
> 🧠 The reasoning shift (2024-2026) - the biggest change since ChatGPT
> Up to GPT-4, a decoder answered in a single pass. **Reasoning models** (OpenAI o1 / o3, GPT-5's thinking mode, DeepSeek-R1) changed that: they spend extra compute *at inference* - writing a long hidden chain-of-thought, exploring several lines of attack, then committing to an answer. More thinking tokens buys more accuracy on math, code, and logic. This is **test-time compute scaling**.
> The training trick is reinforcement learning on *verified* traces. DeepSeek showed it with **GRPO** (group-relative policy optimization - no separate reward model), and that the **R1-Zero** variant learned to reason with pure RL and *no* supervised warm-up. (Deep dive: Module 9.)
> **Engineering consequence:** reasoning models override your hand-tuned temperature / top-p - they ship with *locked* sampling defaults, you pay for the hidden thinking tokens, and latency is higher. Reach for them on hard problems, not on "rewrite this email." Sources: DeepSeek-R1, [arXiv 2501.12948](https://arxiv.org/abs/2501.12948); OpenAI o3.

---

## 🎯 Concept 8: Scaling Laws & Emergent Abilities - Why Bigger Models Are Smarter

### Scaling Laws & Emergent Abilities - Why Bigger Models Are Smarter

Performance follows predictable power laws. But some abilities appear suddenly at scale.

> 📈 **Analogy**
>
> **Scaling laws are like learning curves for AI.** Double the compute → predictable improvement in next-token prediction. But some abilities - arithmetic, translation, chain-of-thought reasoning - appear to "switch on" at certain scales. GPT-3 (175B) could do few-shot arithmetic that GPT-2 (1.5B) completely failed at. This is why labs keep scaling.

**📝 `scaling_laws.py Why Scale`** - *Matters*

In [ ]:
# =============================================
# SCALING LAWS - How size → performance
# =============================================

# Kaplan Scaling Laws (2020) - OpenAI
# "Performance follows power law with model size, data, compute"
# 10x more compute → scale model 5.5x, data 1.8x
# This led to GPT-3: 175B params but only 300B tokens (undertrained!)

# Chinchilla Scaling Laws (2022) - DeepMind
# "Data matters more than we thought!"
# Optimal: 20 tokens per parameter
# GPT-3 should have been either 20x smaller OR trained on 20x more data

scaling = [
    ("GPT-3",     175,   300,    1.7,   "Undertrained by Chinchilla standards"),
    ("Chinchilla", 70,   1400,   20.0,  "Optimal: matched GPT-3 quality at 2.5x less cost"),
    ("LLaMA 1",   7,     1000,   142.0, "Overtrained: better at inference (cheaper to serve)"),
    ("LLaMA 3",   8,     15000,  1875.0,"Massively overtrained: best 8B model ever"),
]

print(f"{'Model':<12} {'Params(B)':>10} {'Tokens(B)':>10} {'Tok/Param':>10} Strategy")
print("-" * 75)
for name, params, tokens, ratio, desc in scaling:
    print(f"  {name:<10} {params:>10} {tokens:>10} {ratio:>10.1f} {desc}")

print("""
EMERGENT ABILITIES - capabilities that "switch on" at scale:
  ~13B params: Few-shot arithmetic, word unscrambling
  ~62B params: Chain-of-thought reasoning
  ~175B params: In-context learning, code generation
  
  But recent research (2023-2025) shows: "emergence" may be a
  measurement artifact - smooth improvement looks discontinuous
  when using binary (exact match) metrics. With continuous metrics,
  improvements are gradual. Still, scale clearly helps.

IN-CONTEXT LEARNING - GPT's superpower:
  Prompt: "Translate: cat→gato, dog→perro, house→"
  Output: "casa"
  
  GPT learned translation from 2 examples IN THE PROMPT.
  No weight updates. No fine-tuning. Just pattern recognition
  from pre-training on billions of text patterns.
  This is why prompt engineering (Module 5) is so powerful.
""")

# Output:
#   GPT-3      175    300    1.7  Undertrained by Chinchilla
#   LLaMA 3      8  15000 1875.0  Massively overtrained

#### 💡 What just happened

What just happened?
**Two scaling law eras:** Kaplan (2020) said "make models bigger." Chinchilla (2022) said "use more data." LLaMA 3 took this further: 8B params trained on 15 TRILLION tokens - 1,875x more data than the "optimal" ratio. This "overtraining" makes small models cheaper to serve. **Emergent abilities** like in-context learning (learning from prompt examples without weight updates) appear at scale - this is why prompt engineering works and why it's covered in Module 5.

---

## 🎯 Concept 9: The 2026 Decoder Landscape - Open vs Closed Models

### The 2026 Decoder Landscape - Open vs Closed Models

The models you'll use in production. Know what's available, what it costs, and when to use each.

**📝 `decoder_landscape.py 2026`** - *Models*

In [ ]:
# =============================================
# 2026 DECODER MODEL LANDSCAPE (prices verified June 2026)
# =============================================

models = [
    # (Name, Provider, Type, Size, $/M input, $/M output, Notes)
    ("GPT-5.5",         "OpenAI",    "Closed", "unknown",  5.00,  30.00, "Unified: fast vs thinking router"),
    ("Claude Opus 4.8", "Anthropic", "Closed", "unknown", 5.00,  25.00, "Top-tier code + agentic runs"),
    ("Claude Sonnet 4.6","Anthropic","Closed","unknown", 3.00,  15.00, "Workhorse: quality per dollar"),
    ("Gemini 3 Flash",  "Google",    "Closed", "unknown", 0.50,  3.00,  "Cheap, fast, 1M context, MM"),
    ("Llama 4 Scout",   "Meta",      "Open",   "17B/16E", 0.00,  0.00,  "Self-host free; 10M context"),
    ("DeepSeek R1/V4", "DeepSeek",  "Open",   "671B/37B",0.55, 2.19,  "Open-weights reasoning, MoE"),
    ("Qwen 3",         "Alibaba",   "Open",   "235B/22B",0.00, 0.00,  "Apache 2.0, multilingual"),
    ("Gemma 3 27B",    "Google",    "Open",   "27B",     0.00,  0.00,  "Best laptop-class open model"),
]

print(f"{'Model':<16} {'Provider':<11} {'Type':<7} {'Size':<10} {'$/M In':>7} {'$/M Out':>8}")
print("-" * 70)
for name, prov, typ, size, inp, out, _ in models:
    print(f"  {name:<14} {prov:<11} {typ:<7} {size:<10} ${inp:>6.2f} ${out:>7.2f}")

print("""
DECISION FRAMEWORK (Module 13 goes deeper):
  Budget app:     Gemini 3 Flash ($0.50/M) - cheap, fast, 1M context
  Best quality:   GPT-5.5 or Claude Opus 4.8 for hard code + reasoning
  Reasoning:      o3 / GPT-5 thinking / DeepSeek R1 - compute at inference
  Self-hosted:    Llama 4 or Qwen 3 (free weights, your GPUs)
  Edge/laptop:    Gemma 3 27B or a quantized Llama 4 Scout
  Indian langs:   Gemini (strong Hindi), Qwen (broad multilingual)
  Fine-tuning:    Llama 4 or Qwen 3 (Module 9)

  Open weights = free to download, run, and fine-tune (you pay compute).
  Closed APIs  = per-token cost, zero infra.
  MoE is everywhere: DeepSeek is 671B total but only 37B active per token.
""")

# Output:
#   GPT-5.5         OpenAI    Closed  $ 5.00  $30.00
#   Gemini 3 Flash  Google    Closed  $ 0.50  $ 3.00

#### 💡 What just happened

What just happened?

  The 2026 decoder landscape: **Closed models** (GPT-5.5, Claude, Gemini) offer convenience at per-token cost. **Open models** (Llama 4, DeepSeek, Qwen, Gemma) are free to download, self-host, and fine-tune. DeepSeek R1/V4 uses MoE - 671B total params but only 37B active per token, making it incredibly cost-effective. **Module 9 (fine-tuning) and Module 13 (cost & performance) build directly on this landscape.**

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 10: Logits & Logprobs - See What GPT Is Thinking

### Logits & Logprobs - See What GPT Is Thinking

The most powerful debugging tool: see the probability of every token at every step.

**📝 `logprobs_explorer.py Debug`** - *Tool*

In [ ]:
# =============================================
# LOGITS → LOGPROBS - See inside the model's head
# =============================================
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

prompt = "The capital of India is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]  # Last position's raw scores

# Logits → probabilities → logprobs
probs = F.softmax(logits, dim=-1)
logprobs = torch.log(probs)

# Top 5 most likely next tokens
top5 = torch.topk(probs, 5)
print("What GPT thinks comes after 'The capital of India is':\n")
for i in range(5):
    token = tokenizer.decode(top5.indices[i])
    prob = top5.values[i].item()
    lp = logprobs[top5.indices[i]].item()
    print(f"  {i+1}. '{token}' → prob={prob:.1%}, logprob={lp:.2f}")

print("""
LOGITS vs LOGPROBS vs PROBABILITIES:
  logits  = raw scores from model (unbounded, e.g. -2.1, 5.3, 1.7)
  probs   = softmax(logits) → sums to 1.0 (e.g. 0.45, 0.30, 0.15)
  logprobs = log(probs) → always negative (e.g. -0.80, -1.20, -1.90)
  
  logprob = 0.0  → 100% confident (never happens in practice)
  logprob = -0.7 → ~50% confident
  logprob = -2.3 → ~10% confident
  logprob = -4.6 → ~1% confident

PRODUCTION USES of logprobs:
  1. Hallucination detection → low logprob on factual claims = flag for review
  2. Classification confidence → use logprobs instead of text parsing
  3. Perplexity = exp(-avg logprob) → standard metric for model quality
     Lower perplexity = model is more certain = better model
  4. Token cost optimization → route low-confidence queries to larger models
""")

# Output:
#   1. ' New'   -> prob=64.2%, logprob=-0.44
#   2. ' Delhi' -> prob=18.1%, logprob=-1.71

#### 💡 What just happened

What just happened?
  You looked inside GPT's decision process. At each generation step, the model produces **logits** (raw scores for all 50,257 tokens), which become **probabilities** via softmax, and **logprobs** via log(). This is the same pipeline your temperature and top-p code modifies! **Perplexity** = exp(-avg logprob) is THE standard metric for evaluating language models. In production, logprobs enable hallucination detection, confidence-based routing, and classification without text parsing.

---

## 🎯 Concept 11: From Raw Model to ChatGPT - The Training Pipeline

### From Raw Model to ChatGPT - The Training Pipeline

The most important story in AI: how a 1.3B InstructGPT beat a 175B GPT-3.

> 🎓 **Analogy**
>
> **Pre-training teaches a student everything in the library. SFT teaches them to answer questions politely. RLHF teaches them what kind of answers humans actually prefer.** A tiny model trained with RLHF (InstructGPT, 1.3B) was preferred over the 175B GPT-3 - alignment beats scale.

**📝 `training_pipeline.py 4`** - *Stages*

In [ ]:
# =============================================
# THE LLM TRAINING PIPELINE - 4 stages
# =============================================

pipeline = {
    "1. Pre-Training": {
        "task": "Next-token prediction on trillions of tokens",
        "data": "Internet text (FineWeb, Common Crawl, books, code)",
        "cost": "$10K-$100M+ (most expensive stage)",
        "output": "Base model - completes text but doesn't follow instructions",
        "example": "'What is 2+2?' → 'What is 2+3? What is 2+4?...' (just continues)",
    },
    "2. SFT (Supervised Fine-Tuning)": {
        "task": "Train on (instruction, ideal_response) pairs",
        "data": "~100K human-written instruction-response examples",
        "cost": "$100-$10K",
        "output": "Instruction-following model - answers questions properly",
        "example": "'What is 2+2?' → '4' (follows instruction format)",
    },
    "3. RLHF / DPO (Alignment)": {
        "task": "Optimize for human preferences",
        "data": "Human rankings: 'Response A is better than B'",
        "cost": "$10K-$1M (human annotators are expensive)",
        "output": "Aligned model - helpful, harmless, honest",
        "methods": "PPO → DPO → GRPO (each simpler than the last)",
    },
    "4. Reasoning (2024+)": {
        "task": "Chain-of-thought fine-tuning + test-time compute",
        "data": "Reasoning traces (verified math, code, logic)",
        "cost": "a large, growing share of the compute budget",
        "output": "Reasoning model - thinks before answering (o1, o3, R1)",
        "key": "DeepSeek R1: GRPO only, no SFT needed for reasoning!",
    },
}

for stage, info in pipeline.items():
    print(f"\n{stage}")
    for k, v in info.items():
        print(f"  {k}: {v}")

print("""
THE INSIGHT: Alignment beats scale.
  InstructGPT (1.3B params + RLHF) was PREFERRED over GPT-3 (175B).
  A small, well-aligned model > a huge, unaligned model.

ALIGNMENT EVOLUTION (each removes a requirement):
  RLHF (2022): Reward model + PPO → 4 models needed
  DPO  (2023): Direct preference → 2 models (no reward model)
  GRPO (2024): Group relative policy → no critic model, lower memory
  ORPO (2024): Single unified objective → 1 model
  
Module 9 covers SFT + RLHF/DPO in depth with hands-on labs.
""")

# Output:
#   1. Pre-Training: next-token prediction on trillions of tokens ...
#   (prints all 4 stages, then 'Alignment beats scale')

#### 💡 What just happened

What just happened?
  The LLM training pipeline has four stages: **pre-training** (learn language), **SFT** (learn to follow instructions), **RLHF/DPO** (learn human preferences), and **reasoning** (learn to think step-by-step). The key insight: **a 1.3B InstructGPT with RLHF was preferred over the 175B GPT-3 - alignment beats scale.** Each new method (PPO→DPO→GRPO→ORPO) is simpler than the last. Module 9 goes deep on stages 2-3.

---

## 🎯 Concept 12: Production Controls - Structured Outputs, Streaming, Chat Templates

### Production Controls - Structured Outputs, Streaming, Chat Templates

The three features you'll use daily as a GenAI engineer.

**📝 `production_controls.py Daily`** - *Tools*

In [ ]:
# =============================================
# PRODUCTION CONTROLS - What engineers use daily
# =============================================
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# 1. CHAT TEMPLATES - system/user/assistant format
# Under the hood, this is just concatenated text for next-token prediction:
#   <system>You are a helpful assistant.</system>
#   <user>What is 2+2?</user>
#   <assistant>4</assistant>
# The model does CLM on this ENTIRE sequence - same as Step 1!
# The system instruction rides in the config below.

# 2. STRUCTURED OUTPUTS - force valid JSON
# Internally: constrained decoding masks non-conforming tokens at each step
# If the schema says "age" must be an integer, tokens like "twenty" get
# probability 0.0 - the model CANNOT produce invalid output.

response = client.models.generate_content(
    model="gemini-3-flash",
    contents="Extract: 'Sanjay Kumar, age 35, lives in Hyderabad'",
    config=types.GenerateContentConfig(
        system_instruction="You are a concise technical assistant. Reply in JSON.",
        response_mime_type="application/json",
        response_schema={
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "integer"},
                "city": {"type": "string"},
            }
        },
    ),
)
print(f"Structured output: {response.text}")
# → {"name": "Sanjay Kumar", "age": 35, "city": "Hyderabad"}

# 3. STREAMING - token-by-token delivery (SSE)
# Why streaming? Lower time-to-first-token; studies link it to longer sessions.
# Each token is sent as it's generated via Server-Sent Events.
for chunk in client.models.generate_content_stream(
    model="gemini-3-flash",
    contents="Explain RAG in 3 bullets",
):
    print(chunk.text, end="", flush=True)  # tokens appear one by one

print("""

OTHER PRODUCTION PARAMETERS:
  repetition_penalty: 1.2  → penalizes tokens already generated (prevents loops)
  frequency_penalty:  0.5  → reduces probability of frequent tokens (OpenAI-specific)
  presence_penalty:   0.5  → encourages topic diversity (OpenAI-specific)
  stop_sequences: ["\\n"]  → stop generating when this token appears
  max_tokens: 500          → hard limit on output length
  
DECODING STRATEGIES (beyond temp/top-k/top-p):
  Greedy:       do_sample=False → always pick highest prob (deterministic)
  Beam search:  num_beams=4 → explore multiple paths (best for translation)
  Contrastive:  penalty_alpha → balances quality and diversity
  Min-p:        min_p=0.1 → dynamic threshold (popular in local LLM community)
""")

#### 💡 What just happened

What just happened?
  Three production essentials: **Chat templates** (system/user/assistant) are just concatenated text for the same CLM you learned in Step 1 - no magic. **Structured outputs** use constrained decoding (masking invalid tokens at each step) to guarantee valid JSON - essential for APIs and agents. **Streaming** via Server-Sent Events delivers tokens one-by-one, sharply improving perceived speed. Plus: repetition penalty, beam search, and stop sequences complete your production toolkit. **Module 10 (Tool Use & MCP) relies heavily on structured outputs for tool calling.**

## The Complete Picture: GPT vs BERT

> ℹ️ **Info**
>
> What we learned - 12 concepts
> - **CLM (Causal Language Modeling)** - GPT learns by predicting the next word, seeing only what came before.
> - **Autoregressive generation** - Generate one word, add it to input, repeat. That's how ALL text generation works.
> - **Temperature** - Divide logits by a number before softmax. Low = safe, high = creative.
> - **Top-k & Top-p** - Top-k: only consider best k words. Top-p: keep until cumulative prob reaches p.
> - **GPT-2 → Gemini** - Same concepts, different scale. You now know what every API parameter does.
> - **GPT family evolution** - GPT-1→2→3→ChatGPT→4→o1→GPT-5: each added a key insight (scale, RLHF, MoE, reasoning).
> - **Scaling laws** - Kaplan: "bigger models." Chinchilla: "more data." LLaMA 3: "overtrain small models for cheap inference."
> - **In-context learning** - GPT learns from prompt examples without weight updates. Foundation of prompt engineering (Module 5).
> - **2026 landscape** - Open (Llama 4, DeepSeek, Qwen) vs Closed (GPT-5, Claude, Gemini). Free to self-host, or roughly $0.50 to $30 per M tokens.
> - **Logits & logprobs** - See the probability of every token. Perplexity = exp(-avg logprob). Enables hallucination detection.
> - **Training pipeline** - Pre-train → SFT → RLHF/DPO → Reasoning. Alignment beats scale (1.3B InstructGPT > 175B GPT-3).
> - **Production controls** - Structured outputs (constrained decoding), streaming (SSE), chat templates, repetition penalty, beam search.

> ℹ️ **Info**
>
> Sources and further reading
> - Nucleus sampling / top-p - Holtzman et al. 2019, [arXiv 1904.09751](https://arxiv.org/abs/1904.09751)
> - Min-p sampling - Nguyen et al. 2024, [arXiv 2407.01082](https://arxiv.org/abs/2407.01082)
> - Chinchilla scaling laws - Hoffmann et al. 2022, [arXiv 2203.15556](https://arxiv.org/abs/2203.15556)
> - InstructGPT (alignment beats scale) - Ouyang et al. 2022, [arXiv 2203.02155](https://arxiv.org/abs/2203.02155)
> - DeepSeek-R1 (open reasoning, GRPO) - Guo et al. 2025, [arXiv 2501.12948](https://arxiv.org/abs/2501.12948)
> - GPT-5 - [openai.com](https://openai.com/index/introducing-gpt-5/); Claude pricing - [platform.claude.com](https://platform.claude.com/docs/en/about-claude/pricing)
> - Hugging Face `generate()` and vLLM implement these decoding controls in production.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **GPT & Decoder Models: Writing One Word at a Time**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-4_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-4.4.html` - regenerate this notebook whenever the source HTML is updated.*
